# 🛡️ Sistema Antifraude Inteligente

### 📌 Contexto de Negócio & Definição do Problema
Em plataformas de e-commerce e fintechs, cada transação carrega dois riscos opostos:
1. **Aprovar uma fraude:** Gera estorno (*chargeback*), penalidades das bandeiras e perda direta de capital.
2. **Bloquear compras legítimas:** Gera atrito desnecessário, perda de receita imediata e insatisfação do cliente (*churn*).

Esse motor traduz as probabilidades estimadas por uma Rede Neural em **faixas operacionais acionáveis**:
* 🟢 **Aprovação Instantânea (Score < 15%):** Risco baixo, fluxo sem fricção para o cliente.
* 🟡 **Desafio de Segurança / 2FA (15% a 65%):** Risco moderado, exige confirmação no app.
* 🔴 **Bloqueio Preventivo (Score > 65%):** Risco crítico, envio direto para contenção e análise manual.

## ⚙️ 1. Preparação do Ambiente e Dependências

Instalamos e importamos as bibliotecas necessárias para manipulação de dados, deep learning com TensorFlow/Keras, avaliação de métricas e construção do simulador interativo com Gradio.

In [ ]:
# Instalação das bibliotecas necessárias
!pip install -q scikit-learn pandas numpy matplotlib seaborn gradio

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve, roc_auc_score, average_precision_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Garantir reprodutibilidade
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow versão: {tf.__version__}")

## 📊 2. Carregamento dos Dados e Diagnóstico do Desbalanceamento

Utilizamos o dataset público de **Credit Card Fraud Detection** via OpenML.

Em problemas de risco e fraude, o desbalanceamento é extremo: a imensa maioria das transações é legítima. Avaliaremos o volume total e a raridade do evento de fraude na base.

In [ ]:
print("Baixando o dataset CreditCardFraud diretamente do TensorFlow/Google Cloud...")

# URL oficial mantida pelo TensorFlow (sem limites de download e sem necessidade de unzip)
url = "https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv"
df = pd.read_csv(url)

# Ajuste da coluna alvo para inteiro binário (0: Legítima, 1: Fraude)
df['Class'] = df['Class'].astype(int)

total_tx = len(df)
fraudes = df['Class'].sum()
legitimas = total_tx - fraudes
taxa_fraude = (fraudes / total_tx) * 100

print(f"\n--- Resumo da Base de Dados ---")
print(f"Total de transações: {total_tx:,}")
print(f"Transações legítimas (Classe 0): {legitimas:,} ({100 - taxa_fraude:.2f}%)")
print(f"Transações fraudulentas (Classe 1): {fraudes:,} ({taxa_fraude:.3f}%)")
print(f"Proporção: 1 fraude para cada {int(legitimas / fraudes)} transações legítimas.")

## 🔧 3. Pré-processamento, Divisão Estratificada e Ponderação de Classes

Nesta etapa:
1. Usamos `RobustScaler` nas variáveis `Amount` e `Time` para evitar que compras de valores discrepantes distorçam os gradientes da rede.
2. Dividimos os dados em **Treino (70%)**, **Validação (15%)** e **Teste (15%)** de forma estratificada para manter rigorosamente a mesma proporção de fraudes em todas as partições.
3. Calculamos `class_weight` para balancear a função de perda durante o treinamento, forçando a rede neural a dar atenção redobrada aos casos de fraude.

In [ ]:
# Escalonamento robusto contra valores discrepantes
scaler_amount = RobustScaler()
scaler_time = RobustScaler()

df['scaled_amount'] = scaler_amount.fit_transform(df['Amount'].values.reshape(-1, 1))
df['scaled_time'] = scaler_time.fit_transform(df['Time'].values.reshape(-1, 1))

# Seleção das features finais (V1 a V28 + colunas escalonadas)
features = [col for col in df.columns if col not in ['Time', 'Amount', 'Class']]
X = df[features].values
y = df['Class'].values

# Divisão estratificada dos dados
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

# Cálculo dos pesos para balancear o aprendizado no Keras
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
weight_for_0 = (1 / neg) * (len(y_train) / 2.0)
weight_for_1 = (1 / pos) * (len(y_train) / 2.0)
class_weights = {0: weight_for_0, 1: weight_for_1}

print(f"Amostras de Treino: {X_train.shape[0]:,} | Validação: {X_val.shape[0]:,} | Teste: {X_test.shape[0]:,}")
print(f"Ponderação -> Classe 0: {weight_for_0:.4f} | Classe 1 (Fraude): {weight_for_1:.4f}")

## 🧠 4. Arquitetura da Rede Neural no Keras

Construímos um classificador profundo multicamadas (MLP) utilizando:
* **BatchNormalization:** Para estabilização e aceleração do treinamento.
* **Dropout:** Para regularização e prevenção de sobreajuste (*overfitting*).
* **Métricas Especializadas:** Monitoramos `PR-AUC` (Área sob a Curva Precision-Recall), métrica de referência para bases desbalanceadas, além de Precision e Recall.
* **EarlyStopping:** Interrupção automática do treino quando o `val_pr_auc` parar de evoluir.

In [ ]:
def build_fraud_model(input_dim):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(64, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),

        layers.Dense(32, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.2),

        layers.Dense(16, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='binary_crossentropy',
        metrics=[
            keras.metrics.AUC(curve='PR', name='pr_auc'),
            keras.metrics.AUC(curve='ROC', name='roc_auc'),
            keras.metrics.Precision(name='precision'),
            keras.metrics.Recall(name='recall')
        ]
    )
    return model

model = build_fraud_model(X_train.shape[1])
model.summary()

# Callback para restauração dos melhores pesos com base em PR-AUC
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_pr_auc',
    mode='max',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    batch_size=1024,
    epochs=30,
    class_weight=class_weights,
    callbacks=[early_stopping],
    verbose=1
)

## 📈 5. Curvas de Aprendizado e Convergência

Avaliamos a estabilidade do treino comparando a curva de perda (*loss*) e a área sob a curva Precision-Recall (`PR-AUC`) entre treino e validação.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Evolução da Loss
ax[0].plot(history.history['loss'], label='Treino')
ax[0].plot(history.history['val_loss'], label='Validação', linestyle='--')
ax[0].set_title('Evolução da Perda (Binary Crossentropy)', fontsize=12)
ax[0].set_xlabel('Épocas')
ax[0].set_ylabel('Loss')
ax[0].legend()
ax[0].grid(True, alpha=0.3)

# Evolução do PR-AUC
ax[1].plot(history.history['pr_auc'], label='Treino')
ax[1].plot(history.history['val_pr_auc'], label='Validação', linestyle='--')
ax[1].set_title('Evolução de PR-AUC (Validação)', fontsize=12)
ax[1].set_xlabel('Épocas')
ax[1].set_ylabel('PR-AUC')
ax[1].legend()
ax[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 🎯 5.1. Avaliação Detalhada: Matriz de Confusão

**O que estamos analisando aqui:**  
Avaliamos a capacidade do modelo de discriminar entre transações legítimas e fraudulentas no conjunto de teste:
* **Verdadeiros Negativos (TN):** Compras legítimas aprovadas normalmente.
* **Falsos Positivos (FP):** Compras legítimas classificadas como fraude (atrito desnecessário com o cliente).
* **Falsos Negativos (FN):** Fraudes reais que o modelo deixou passar (geram estorno/chargeback).
* **Verdadeiros Positivos (TP):** Fraudes interceptadas com sucesso pela rede neural.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Limiar de decisão para a matriz de referência (50%)
y_pred_binario = (y_pred_probs >= 0.50).astype(int)

# Cálculo da matriz de confusão
classes_nomes = ['Legítima (0)', 'Fraude (1)']
cm = confusion_matrix(y_test, y_pred_binario)

# Configuração da figura com o mesmo estilo visual da aula
fig, ax = plt.subplots(figsize=(8, 7), dpi=100)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=classes_nomes
)

# Plotagem com mapa de cores Blues e formatação numérica com separador de milhar
disp.plot(
    cmap=plt.cm.Blues,
    values_format=',d',
    ax=ax,
    colorbar=True
)

# Ajuste fino da tipografia e rotação dos rótulos
ax.set_title('Matriz de Confusão', fontsize=14, pad=15)
ax.set_xlabel('Classe Prevista', fontsize=12, labelpad=10)
ax.set_ylabel('Classe Real', fontsize=12, labelpad=10)
plt.xticks(rotation=45, ha='right', fontsize=11)
plt.yticks(fontsize=11)

# Ajustar tamanho das fontes dentro dos quadrados
for text in disp.text_.ravel():
    text.set_fontsize(13)

plt.tight_layout()
plt.show()

## 🚦 6. Motor de Decisão Antifraude (A Camada de Produto)

Aqui conectamos a inferência estatística ao fluxo operacional do negócio. O score probabilístico é convertido em faixas de decisão (*Risk Bands*):

| Faixa de Risco | Score | Ação Operacional | Impacto no Usuário |
| :--- | :---: | :--- | :--- |
| **Baixo Risco** | `< 15%` | `AUTO_APROVADO` | Zero atrito, compra concluída em milissegundos |
| **Risco Moderado** | `15% a 65%` | `DESAFIO_2FA` | Solicita biometria/SMS no app do titular |
| **Alto Risco** | `> 65%` | `BLOQUEIO_PREVENTIVO` | Bloqueio imediato da cobrança e envio para auditoria |

In [ ]:
# Predições no conjunto de teste independente
y_pred_probs = model.predict(X_test, batch_size=1024).flatten()

# Motor de decisão por faixas operacionais
def motor_de_decisao(prob):
    if prob < 0.15:
        return 'AUTO_APROVADO'
    elif prob <= 0.65:
        return 'DESAFIO_2FA'
    else:
        return 'BLOQUEIO_PREVENTIVO'

df_teste = pd.DataFrame({
    'Classe_Real': y_test,
    'Score_Risco': y_pred_probs,
    'Decisao': [motor_de_decisao(p) for p in y_pred_probs]
})

print("--- Distribuição das Ações Operacionais no Conjunto de Teste ---")
distribuicao = df_teste['Decisao'].value_counts(normalize=True) * 100
for categoria, pct in distribuicao.items():
    print(f"• {categoria}: {pct:.2f}% do total de transações")

## 💰 7. Relatório de Impacto Financeiro e ROI de Negócio

Para comprovar o valor do Produto de IA, quantificamos os ganhos e custos em reais:
* **Prejuízo Evitado:** Fraudes barradas diretamente ou capturadas via desafio 2FA (Ticket Médio + Taxa de Estorno/Chargeback).
* **Custo Operacional & Atrito:** Custo de disparo de 2FA somado à perda potencial de clientes fiéis bloqueados indevidamente (*falso positivo*).

In [ ]:
# Premissas de Negócio e Custos de Mercado
TICKET_MEDIO = 420.00
CUSTO_CHARGEBACK = 75.00
CUSTO_SMS_2FA = 0.15
TAXA_CHURN_FALSO_POSITIVO = 0.04  # 4% dos clientes legítimos bloqueados abandonam o serviço

total_fraudes = (df_teste['Classe_Real'] == 1).sum()
total_legitimas = (df_teste['Classe_Real'] == 0).sum()

# Fraudes contidas
fraudes_bloqueadas = ((df_teste['Classe_Real'] == 1) & (df_teste['Decisao'] == 'BLOQUEIO_PREVENTIVO')).sum()
fraudes_em_2fa = ((df_teste['Classe_Real'] == 1) & (df_teste['Decisao'] == 'DESAFIO_2FA')).sum()
fraudes_resolvidas_2fa = int(fraudes_em_2fa * 0.90)  # 90% dos fraudadores não passam no 2FA
fraudes_escapadas = total_fraudes - (fraudes_bloqueadas + fraudes_resolvidas_2fa)

# Atrito com clientes legítimos
legitimos_bloqueados = ((df_teste['Classe_Real'] == 0) & (df_teste['Decisao'] == 'BLOQUEIO_PREVENTIVO')).sum()
desafios_2fa_enviados = (df_teste['Decisao'] == 'DESAFIO_2FA').sum()

# Demonstrativo de Resultados
prejuizo_evitado = (fraudes_bloqueadas + fraudes_resolvidas_2fa) * (TICKET_MEDIO + CUSTO_CHARGEBACK)
prejuizo_residual = fraudes_escapadas * (TICKET_MEDIO + CUSTO_CHARGEBACK)
custo_operacional = (desafios_2fa_enviados * CUSTO_SMS_2FA) + (legitimos_bloqueados * TICKET_MEDIO * TAXA_CHURN_FALSO_POSITIVO)
resultado_liquido = prejuizo_evitado - custo_operacional

print("="*60)
print("              RELATÓRIO FINANCEIRO EXECUTIVO")
print("="*60)
print(f"Fraudes Interceptadas: {fraudes_bloqueadas + fraudes_resolvidas_2fa} de {total_fraudes} ({(fraudes_bloqueadas + fraudes_resolvidas_2fa)/total_fraudes*100:.1f}%)")
print(f"Transações Legítimas sem Atrito: {((df_teste['Classe_Real'] == 0) & (df_teste['Decisao'] == 'AUTO_APROVADO')).sum():,} ({((df_teste['Classe_Real'] == 0) & (df_teste['Decisao'] == 'AUTO_APROVADO')).sum()/total_legitimas*100:.2f}%)")
print(f"Falsos Bloqueios Críticos: {legitimos_bloqueados}")
print("-" * 60)
print(f"💸 Prejuízo Bruto Evitado:     R$ {prejuizo_evitado:,.2f}")
print(f"⚠️ Prejuízo Residual:          R$ {prejuizo_residual:,.2f}")
print(f"📉 Custo Operacional e Atrito: R$ {custo_operacional:,.2f}")
print(f"✅ Economia Líquida Gerada:    R$ {resultado_liquido:,.2f}")
print("="*60)

## 🖥️ 8. Simulador Operacional Interativo (Gradio)

Interface visual embutida no notebook para permitir a validação prática do motor de decisão por analistas de risco, gestores ou durante a apresentação.

In [ ]:
import gradio as gr

# Fecha instâncias anteriores para liberar portas
gr.close_all()

def processar_analise(valor, hora_do_dia, v1, v2, v3, v4):
    # Conversão: transforma a hora informada (0 a 24h) em segundos para o scaler
    segundos_dia = hora_do_dia * 3600

    entrada = np.zeros((1, X_train.shape[1]))
    entrada[0, -2] = scaler_amount.transform([[valor]])[0][0]
    entrada[0, -1] = scaler_time.transform([[segundos_dia]])[0][0]
    entrada[0, 0] = v1
    entrada[0, 1] = v2
    entrada[0, 2] = v3
    entrada[0, 3] = v4

    score = float(model.predict(entrada, verbose=0)[0][0])
    decisao = motor_de_decisao(score)

    if decisao == 'AUTO_APROVADO':
        badge = "<div style='background-color:#0f5132; color:#d1e7dd; padding:16px; border-radius:10px; text-align:center; font-size:18px; font-weight:bold;'>🟢 APROVADO AUTOMATICAMENTE</div>"
        instrucao = "Transação com baixo padrão de anomalia. Liberada sem atrito com o cliente."
    elif decisao == 'DESAFIO_2FA':
        badge = "<div style='background-color:#664d03; color:#fff3cd; padding:16px; border-radius:10px; text-align:center; font-size:18px; font-weight:bold;'>🟡 DESAFIO DE SEGURANÇA (2FA)</div>"
        instrucao = "Risco moderado detectado. Solicitar validação biométrica ou token SMS no app."
    else:
        badge = "<div style='background-color:#842029; color:#f8d7da; padding:16px; border-radius:10px; text-align:center; font-size:18px; font-weight:bold;'>🔴 BLOQUEIO PREVENTIVO</div>"
        instrucao = "Risco crítico de fraude detectado. Pagamento barrado e enviado à mesa de risco nível 2."

    gauge_html = f"""
    <div style='background:#1e293b; padding:16px; border-radius:12px; margin-bottom:12px; border: 1px solid #334155;'>
        <p style='color:#94a3b8; margin:0 0 4px 0; font-size:13px; font-weight:600;'>PROBABILIDADE DE FRAUDE ESTIMADA</p>
        <p style='font-size:32px; font-weight:800; color:#f8fafc; margin:0;'>{score:.2%}</p>
    </div>
    """

    return gauge_html, badge, instrucao

with gr.Blocks() as demo:
    gr.Markdown(
        """
        # 🛡️ Central de Decisão Antifraude & Risco Operacional
        **Produto de IA:** Avaliação em tempo real conectando modelo Keras a políticas de esteira operacional.
        """
    )

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 📥 Dados da Transação")
            valor = gr.Number(value=350.0, label="Valor da Transação (R$)")

            # Slider agora configurado de 0h a 23h (passo de 1 hora)
            horario = gr.Slider(
                minimum=0,
                maximum=23,
                value=14,
                step=1,
                label="Horário da Compra (Hora do Dia: 0h a 23h)",
                info="Ex: 3 = 03h00 da madrugada | 14 = 14h00 da tarde"
            )

            gr.Markdown("#### Padrões Comportamentais (Features Anônimas)")
            v1 = gr.Slider(-5.0, 5.0, value=-0.2, step=0.1, label="V1: Desvio de Localidade")
            v2 = gr.Slider(-5.0, 5.0, value=0.1, step=0.1, label="V2: Padrão do Dispositivo")
            v3 = gr.Slider(-5.0, 5.0, value=0.0, step=0.1, label="V3: Velocidade de Preenchimento")
            v4 = gr.Slider(-5.0, 5.0, value=0.3, step=0.1, label="V4: Frequência Recente")

            btn_analisar = gr.Button("⚡ Avaliar Transação no Motor", variant="primary")

        with gr.Column(scale=1):
            gr.Markdown("### 🎯 Decisão da Esteira de Risco")
            saida_score = gr.HTML("<div style='background:#1e293b; padding:16px; border-radius:12px; color:#94a3b8;'>Aguardando avaliação...</div>")
            saida_badge = gr.HTML("<div style='background:#1e293b; padding:16px; border-radius:10px; color:#94a3b8; text-align:center;'>Status</div>")
            saida_diretriz = gr.Textbox(label="Diretriz Operacional para o Analista", interactive=False)

            gr.Markdown("### 🧪 Casos Rápidos para Teste")
            gr.Examples(
                examples=[
                    [89.50, 14, 0.1, -0.2, 0.3, -0.1],     # Compra comum às 14h00
                    [1850.00, 3, -2.8, 3.5, -3.1, 2.9],    # Compra alta às 03h00 (madrugada suspeita)
                    [420.00, 2, -1.2, 1.8, -1.5, 1.1]      # Suspeita moderada às 02h00
                ],
                inputs=[valor, horario, v1, v2, v3, v4],
                label="Clique para carregar um cenário pronto:"
            )

    btn_analisar.click(
        fn=processar_analise,
        inputs=[valor, horario, v1, v2, v3, v4],
        outputs=[saida_score, saida_badge, saida_diretriz]
    )

demo.launch(theme=gr.themes.Soft(primary_hue="emerald", neutral_hue="slate"), share=True, inline=True)
